# 립리딩 전처리 · 학습 파이프라인 (Colab)

원본 영상을 Google Drive에 두고, 코랩 GPU로 전처리와 학습을 수행한다.

**실행 전 준비**
1. 런타임 → 런타임 유형 변경 → 하드웨어 가속기 **GPU** 선택
2. Drive에 영상 폴더 생성 후 녹화본 업로드
3. 파일명 규칙: `{화자}_{문구}_{번호}.mp4` — 예) `s01_물주세요_01.mp4`

화자가 **2명 이상**이어야 학습이 진행된다. 화자 단위로 학습·검증을 나누기 때문이다.

## 1. 환경 확인

In [10]:
import torch

print(f"torch {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("경고: 런타임 유형을 GPU로 변경하세요.")

torch 2.11.0+cu128
CUDA 사용 가능: True
GPU: NVIDIA A100-SXM4-40GB


## 2. Drive 마운트

In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 3. 경로 설정

`DRIVE_ROOT` 아래 `raw/`에 영상을 두면, 전처리 결과가 `processed/`에 저장된다.
Drive에 저장하므로 런타임이 끊겨도 전처리를 다시 하지 않아도 된다.

In [5]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_RAW = DRIVE_ROOT / "raw"
DRIVE_PROCESSED = DRIVE_ROOT / "processed"
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"

for folder in (DRIVE_RAW, DRIVE_PROCESSED, DRIVE_CHECKPOINTS):
    folder.mkdir(parents=True, exist_ok=True)

videos = sorted(
    p.name for p in DRIVE_RAW.glob("*") if p.suffix.lower() in (".mp4", ".avi", ".mov")
)
print(f"영상 {len(videos)}개")
for name in videos[:10]:
    print(f"  {name}")

영상 1234개
  s01_가래가있어요_01.mp4
  s01_가래가있어요_02.mp4
  s01_가래가있어요_03.mp4
  s01_가래가있어요_04.mp4
  s01_가래가있어요_05.mp4
  s01_가래가있어요_06.mp4
  s01_가래가있어요_07.mp4
  s01_가래가있어요_08.mp4
  s01_가래가있어요_09.mp4
  s01_가래가있어요_10.mp4


## 4. 저장소 clone

이미 받아둔 저장소가 있으면 `git pull`로 최신 코드만 가져온다.
clone이 중간에 실패해 빈 폴더가 남은 경우에는 지우고 다시 받는다.

비공개 저장소면 `https://<TOKEN>@github.com/...` 형태로 토큰을 넣는다.
토큰은 노트북에 저장하지 말고 매번 입력한다.

**코드를 갱신한 뒤에는 런타임을 다시 시작하거나 모듈을 reload해야 반영된다.**
파이썬은 한 번 import한 모듈을 다시 읽지 않는다.

```python
import importlib
import src.ml.training.dataset, src.ml.training.train
importlib.reload(src.ml.training.dataset)
importlib.reload(src.ml.training.train)
from src.ml.training.train import train
```

In [6]:
import os

REPO_URL = "https://github.com/HumanRhoid/hanium-lipreading.git"
BRANCH = "develop"
REPO_DIR = Path("/content/hanium-lipreading")

# clone이 중간에 실패하면 빈 폴더만 남아 다음 실행에서 git 명령이 어긋난다.
os.chdir("/content")
if (REPO_DIR / ".git").exists():
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
else:
    !rm -rf {REPO_DIR}
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"작업 경로: {Path.cwd()}")

Cloning into '/content/hanium-lipreading'...
remote: Enumerating objects: 671, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 671 (delta 98), reused 110 (delta 62), pack-reused 441 (from 1)
Receiving objects: 100% (671/671), 717.52 KiB | 7.40 MiB/s, done.
Resolving deltas: 100% (302/302), done.
작업 경로: /content/hanium-lipreading


## 5. 의존성 설치

`uv sync`는 쓰지 않는다. `pyproject.toml`이 torch를 **CPU 전용 인덱스**로 고정하고 있어
코랩의 GPU torch가 CPU 버전으로 교체되기 때문이다. 필요한 것만 pip로 설치한다.

In [7]:
!pip install --quiet mediapipe opencv-python wandb

import torch

print(f"설치 후 CUDA 사용 가능: {torch.cuda.is_available()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 14.6 MB/s eta 0:00:00
설치 후 CUDA 사용 가능: True


## 6. 얼굴 랜드마크 모델 내려받기

`face_landmarker.task`는 `.gitignore`에 제외돼 있어 저장소에 없다.

In [8]:
LANDMARKER_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
landmarker_path = REPO_DIR / "models" / "face_landmarker.task"
landmarker_path.parent.mkdir(parents=True, exist_ok=True)

if not landmarker_path.exists():
    !wget -q -O {landmarker_path} {LANDMARKER_URL}

print(f"{landmarker_path.name}: {landmarker_path.stat().st_size / 1e6:.1f} MB")

face_landmarker.task: 3.8 MB


## 7. 전처리 — 영상을 .npy로 변환

Drive를 입출력으로 직접 지정한다. 이미 변환된 파일은 건너뛴다.

In [15]:
import sys

sys.path.insert(0, str(REPO_DIR))

from src.ml.preprocess.vid2npy import run_batch

run_batch(raw_dir=DRIVE_RAW, processed_dir=DRIVE_PROCESSED)

이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_01.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_02.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_03.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_04.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_05.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_06.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_07.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_08.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_09.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_10.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_11.npy
이미 존재함, 건너뜀: /content

In [16]:
FRAMES = 45
PROCESSED_F45 = DRIVE_ROOT / "processed_f45"

from src.ml.preprocess import vid2npy
from src.ml.preprocess.lip_crop import crop_lip_frames
from src.ml.preprocess.normalize import normalize_frames

# normalize_frames의 기본값은 def 시점에 고정되므로 함수를 갈아끼운다
def process_video_f45(video_path, landmarker):
    lips, opennesses = crop_lip_frames(video_path, landmarker)
    if not lips:
        return None
    return normalize_frames(lips, opennesses, fixed_frame_count=FRAMES)

vid2npy.process_video = process_video_f45
vid2npy.run_batch(raw_dir=DRIVE_RAW, processed_dir=PROCESSED_F45)

이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_01.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_02.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_03.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_04.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_05.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_06.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_07.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_08.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_09.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_10.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s0

In [17]:
FRAMES = 60
PROCESSED_F60 = DRIVE_ROOT / "processed_f60"

from src.ml.preprocess import vid2npy
from src.ml.preprocess.lip_crop import crop_lip_frames
from src.ml.preprocess.normalize import normalize_frames

def process_video_f60(video_path, landmarker):
    lips, opennesses = crop_lip_frames(video_path, landmarker)
    if not lips:
        return None
    return normalize_frames(lips, opennesses, fixed_frame_count=FRAMES)

vid2npy.process_video = process_video_f60
vid2npy.run_batch(raw_dir=DRIVE_RAW, processed_dir=PROCESSED_F60)

이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_01.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_02.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_03.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_04.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_05.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_06.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_07.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_08.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_09.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_10.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s0

## 8. 매니페스트 생성

`.npy` 파일명을 파싱해 라벨과 화자를 뽑아낸다.

In [18]:
from scripts.build_manifest import build

manifest_path = DRIVE_ROOT / "manifest.csv"
build(processed_dir=DRIVE_PROCESSED, manifest_path=manifest_path)

매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest.csv
  클립 1234개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요


In [19]:
import csv
from collections import Counter

with open(manifest_path, encoding="utf-8") as manifest_file:
    rows = list(csv.DictReader(manifest_file))

print(f"클립 {len(rows)}개")
print(f"화자별: {dict(Counter(row['speaker_id'] for row in rows))}")
print(f"문구별: {dict(Counter(row['label_text'] for row in rows))}")

클립 1234개
화자별: {'s01': 157, 's03': 148, 's04': 150, 's05': 150, 's06': 157, 's07': 171, 's08': 151, 's09': 150}
문구별: {'가래가있어요': 80, '간호사불러주세요': 79, '더워요': 86, '도와주세요': 82, '물주세요': 88, '배고파요': 85, '보호자불러주세요': 82, '숨쉬기힘들어요': 82, '아파요': 79, '어지러워요': 80, '자세바꿔주세요': 83, '진통제주세요': 86, '추워요': 82, '토할거같아요': 80, '화장실가고싶어요': 80}


### (45 프레임)


In [20]:
from scripts.build_manifest import build

manifest_f45 = DRIVE_ROOT / "manifest_f45.csv"
build(processed_dir=PROCESSED_F45, manifest_path=manifest_f45)

매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest_f45.csv
  클립 1234개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요


### (60프레임 매니페스트 및 로컬 복사 및 검증포함)


In [21]:
from scripts.build_manifest import build
from pathlib import Path
import shutil, time, numpy as np

manifest_f60 = DRIVE_ROOT / "manifest_f60.csv"
build(processed_dir=PROCESSED_F60, manifest_path=manifest_f60)

LOCAL_F60 = Path("/content/data60")
target = LOCAL_F60 / "processed"
expected = len(list(PROCESSED_F60.glob("*.npy")))

if target.exists() and len(list(target.glob("*.npy"))) != expected:
    shutil.rmtree(target)
if not target.exists():
    t = time.time()
    shutil.copytree(PROCESSED_F60, target)
    print(f"복사 {time.time() - t:.0f}초")

# 아까 겪은 0바이트·누락 확인
bad = [p.name for p in target.glob("*.npy")
       if p.stat().st_size == 0 or (np.load(p, mmap_mode="r") is None)]
n = len(list(target.glob("*.npy")))
print(f"로컬 {n}/{expected}개 · 손상 {len(bad)}개")
assert n == expected and not bad, "복사가 불완전합니다"

TRAIN_ROOT_F60 = LOCAL_F60

매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest_f60.csv
  클립 1234개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요
복사 34초
로컬 1234/1234개 · 손상 0개


## 8-1. 학습 데이터를 로컬 디스크로 복사

Drive 마운트는 네트워크 파일시스템이라 매 에폭 수백 개를 원격에서 읽는다.
런타임 로컬 디스크로 옮기면 GPU가 데이터를 기다리는 시간이 줄어든다.

런타임이 끊기면 사라지므로 세션마다 다시 실행한다. 복사에 1~2분 걸린다.

In [22]:
import shutil
import time

LOCAL_ROOT = Path("/content/data")
LOCAL_PROCESSED = LOCAL_ROOT / "processed"

started = time.time()
if not LOCAL_PROCESSED.exists():
    shutil.copytree(DRIVE_PROCESSED, LOCAL_PROCESSED)

# 매니페스트의 clip_path가 "processed/..." 라서 data_root만 바꾸면 그대로 맞는다.
TRAIN_ROOT = LOCAL_ROOT
local_count = len(list(LOCAL_PROCESSED.glob("*.npy")))
print(f"로컬 npy {local_count}개 · {time.time() - started:.0f}초")

로컬 npy 1234개 · 40초


### (45 프레임)


In [23]:
import shutil, time

LOCAL_F45 = Path("/content/data45")
target = LOCAL_F45 / "processed"      # 매니페스트가 "processed/..." 로 적으므로 이 이름이어야 함
expected = len(list(PROCESSED_F45.glob("*.npy")))

if target.exists() and len(list(target.glob("*.npy"))) != expected:
    shutil.rmtree(target)             # 개수 안 맞으면 다시 복사 (8-1 셀의 그 함정)
if not target.exists():
    started = time.time()
    shutil.copytree(PROCESSED_F45, target)
    print(f"복사 {time.time() - started:.0f}초")

TRAIN_ROOT_F45 = LOCAL_F45
print(f"로컬 npy {len(list(target.glob('*.npy')))}개 / 기대 {expected}개")

복사 48초
로컬 npy 1234개 / 기대 1234개


## 9. 학습

체크포인트는 Drive에 저장되므로 런타임이 끊겨도 남는다.
학습 데이터는 `TRAIN_ROOT`(로컬 복사본)에서 읽어 I/O 대기를 줄인다.

**실험은 이 셀의 인자만 바꾸면 된다.** 저장소 코드를 고칠 필요가 없다.

- `seed` — 가중치 초기값·데이터 순서·증강을 한꺼번에 고정한다.
  같은 시드면 같은 결과가 나오므로 설정 비교의 전제가 된다
- `val_speakers` — 검증에 쓸 화자. **설정을 비교할 때는 반드시 고정한다.**
  `None`이면 화자 구성이 바뀔 때 검증 대상도 함께 바뀌어 비교가 깨진다
- `hidden_dim` / `num_layer` / `dropout` — 모델 크기와 정규화 강도
- `weight_decay` — 가중치를 작게 유지해 과적합을 억제
- `smoothing` — 최근 몇 에폭 평균으로 체크포인트를 판정할지. `1`이면 단일 에폭 최고치
- `label_smoothing` — 정답 확률을 100%로 몰지 않게 해 과신을 줄인다
- `grad_clip` — 드물게 튀는 그래디언트가 가중치를 흔드는 것을 막는다
- `ema_decay` — 가중치 이동평균으로 검증한다. 후반 진동이 완만해진다
- `augment` / `augmentation_config` — 증강 사용 여부와 강도
- `pretrained` — ImageNet 가중치로 백본을 초기화. 입력 정규화도 함께 바뀐다
- `freeze_backbone` — ResNet 층을 고정. `pretrained`와 함께 쓴다
- `amp` — bfloat16 혼합정밀도. GPU에서만 켜지고 속도가 2~3배 빨라진다
- `wandb_project` — 지정하면 실험이 웹 대시보드에 자동 기록된다

`label_smoothing` · `grad_clip` · `ema_decay`는 안정화 장치다. 셋 다 `0`을 주면
꺼진다. 체크포인트는 EMA를 켜면 평균 가중치로 저장되므로 검증 수치와 일치한다.

**시드 하나로 낸 결과는 그 자체로 성능이 아니다.** 같은 설정이라도 시드가 다르면
0.05~0.1 정도 흔들린다. 설정을 비교하거나 최종 수치를 낼 때는 시드 2~3개로
돌려 평균을 쓴다.

`pretrained=True`에 `freeze_backbone=False`면 학습률을 `3e-5`로 낮춘다. 좋은
초기값을 큰 보폭이 흐트러뜨리기 때문이다. 반대로 동결하면 움직이는 파라미터가
적어 `3e-4`까지 올려도 안정적이다.

**한 번에 하나만 바꾼다.** 두 개를 동시에 바꾸면 무엇이 효과였는지 알 수 없다.
`run_name`에 설정을 알아볼 수 있는 이름을 붙이면 나중에 비교하기 쉽다.

In [ ]:
from src.ml.preprocess.augmentation import AugmentationConfig
from src.ml.training.train import train

# 증강 강도를 조절하려면 설정을 만들어 넘긴다. None이면 기본값을 쓴다.
strong_augmentation = AugmentationConfig(
    brightness_probability=0.7,
    contrast_probability=0.7,
    rotation_probability=0.6,
    shift_probability=0.6,
    zoom_probability=0.6,
)

best_accuracy = train(
    manifest_path=manifest_path,
    data_root=TRAIN_ROOT,
    epochs=80,
    batch_size=16,
    learning_rate=2e-4,
    seed=42,  # 가중치 초기값·데이터 순서·증강을 함께 고정한다
    val_speakers=["s04"],  # 설정 비교 시 고정. None이면 seed로 무작위 선택
    checkpoint_path=DRIVE_CHECKPOINTS / "best.pt",
    num_workers= 8,
    amp=True,
    hidden_dim=300,
    num_layer=2,
    dropout=0.3,
    weight_decay=0.01,
    smoothing=3,
    label_smoothing=0.1,  # 0이면 끔. 정답에 대한 과신을 줄인다
    grad_clip=1.0,  # 0이면 끔. 튀는 그래디언트를 잘라낸다
    ema_decay=0.998,  # 0이면 끔. 가중치 이동평균으로 검증한다
    augment=True,
    augmentation_config=None,  # strong_augmentation 으로 바꿔 강도 실험
    pretrained=False,  # True면 ImageNet 가중치 + ImageNet 입력 정규화
    freeze_backbone=False,  # pretrained와 함께 켜면 ResNet 층을 고정한다
    wandb_project="lipreading",
    run_name="baseline_s04",
)

### (45 프레임)


In [ ]:
from src.ml.training.train import train

SEEDS = [42, 1, 7]
results45 = {}

for seed in SEEDS:
    print(f"\n{'='*16} s06 · seed {seed} · {FRAMES}프레임 {'='*16}")
    results45[seed] = train(
        manifest_path=manifest_f45,
        data_root=TRAIN_ROOT_F45,
        epochs=80,
        batch_size=16,
        learning_rate=2e-4,
        seed=seed,
        val_speakers=["s06"],
        checkpoint_path=DRIVE_CHECKPOINTS / f"f45_s06_seed{seed}.pt",
        num_workers=8,
        amp=True,
        ema_decay=0.998,
        hidden_dim=300,
        num_layer=2,
        dropout=0.3,
        smoothing=3,
        wandb_project="lipreading",
        run_name=f"f45_s06_seed{seed}",
    )

values = [results45[s] for s in SEEDS]
print(f"\n{'='*50}")
print(f"45프레임   {[f'{v:.3f}' for v in values]}")
print(f"           평균 {sum(values)/len(values):.3f} · 폭 {max(values)-min(values):.3f}")
print(f"30프레임   평균 0.754 · 폭 0.051")
print(f"판정선     0.804     ← 넘으면 채택")

### (60 프레임)

In [ ]:
# ═══ 60프레임 8화자 교차검증 · seed 42 ═══
from src.ml.training.train import train

SEEDS = [42, 1, 7]
speakers = sorted({row["speaker_id"] for row in rows})
print(f"화자 {speakers} · 시드 {SEEDS} · 60프레임\n")

results60cv = {}
for speaker in speakers:
    for seed in SEEDS:
        print(f"\n{'='*16} {speaker} · seed {seed} · 60프레임 {'='*16}")
        results60cv[(speaker, seed)] = train(
            manifest_path=manifest_f60,          # ← 60프레임
            data_root=TRAIN_ROOT_F60,
            epochs=80,
            batch_size=16,
            learning_rate=2e-4,
            seed=seed,
            val_speakers=[speaker],
            checkpoint_path=DRIVE_CHECKPOINTS / f"cv60_{speaker}_seed{seed}.pt",
            num_workers=8,
            amp=True,
            ema_decay=0.998,
            hidden_dim=300,
            num_layer=2,
            dropout=0.3,
            smoothing=3,
            wandb_project="lipreading",
            run_name=f"cv60_{speaker}_seed{seed}",
        )

BASE30 = {"s01": 0.656, "s03": 0.446, "s04": 0.780, "s05": 0.307,
          "s06": 0.732, "s07": 0.351, "s08": 0.556, "s09": 0.393}

print(f"\n{'='*56}")
print(f"{'화자':<6}{'30프레임':>10}{'60프레임':>10}{'차이':>10}")
print("-" * 40)
per = {}
for sp in speakers:
    v = sum(results60cv[(sp, s)] for s in SEEDS) / len(SEEDS)
    per[sp] = v
    print(f"{sp:<6}{BASE30[sp]:>10.3f}{v:>10.3f}{v - BASE30[sp]:>+10.3f}")

new, old = sum(per.values()) / len(per), sum(BASE30.values()) / len(BASE30)
print("-" * 40)
print(f"{'평균':<6}{old:>10.3f}{new:>10.3f}{new - old:>+10.3f}")
print(f"\n화자 간 편차   30프레임 {max(BASE30.values())-min(BASE30.values()):.3f}"
      f" · 60프레임 {max(per.values())-min(per.values()):.3f}")

## 9-1. 교차검증

검증 화자 한 명으로 재면 그 사람의 난이도에 결과가 좌우된다. 화자를 바꿔가며
전부 한 번씩 검증으로 쓰고 평균을 내면 화자 편차에 흔들리지 않는 수치가 나온다.

**시드도 함께 반복해야 한다.** 같은 설정이라도 시드가 다르면 0.05~0.1 흔들리는데,
이 폭이 화자 간 차이와 비슷해서 한 번씩만 돌리면 순위를 신뢰할 수 없다.

`SEEDS`를 늘릴수록 신뢰도가 올라가지만 학습 횟수가 화자 수만큼 곱해진다.
경향만 볼 때는 시드 하나로, 발표에 쓸 최종 수치는 셋으로 돌린다.

In [ ]:
SEEDS = [42]  # [42, 1, 7] 로 늘리면 화자마다 여러 번 돌려 편차까지 본다

speakers = sorted({row["speaker_id"] for row in rows})
print(f"화자 {speakers} · 시드 {SEEDS}")

results = {}
for speaker in speakers:
    for seed in SEEDS:
        print(f"\n{'=' * 18} {speaker} · seed {seed} {'=' * 18}")
        results[(speaker, seed)] = train(
            manifest_path=manifest_path,
            data_root=TRAIN_ROOT,
            epochs=80,
            batch_size=16,
            learning_rate=2e-4,
            seed=seed,
            val_speakers=[speaker],
            checkpoint_path=DRIVE_CHECKPOINTS / f"cv_{speaker}_seed{seed}.pt",
            num_workers=8,
            amp=True,
            ema_decay=0.998,
            hidden_dim=300,
            num_layer=2,
            dropout=0.3,
            smoothing=3,
            wandb_project="lipreading",
            run_name=f"cv_{speaker}_seed{seed}",
        )

print(f"\n{'=' * 50}")
per_speaker = {}
for speaker in speakers:
    values = [results[(speaker, s)] for s in SEEDS]
    per_speaker[speaker] = sum(values) / len(values)
    detail = " ".join(f"{v:.3f}" for v in values)
    spread = f" (폭 {max(values) - min(values):.3f})" if len(values) > 1 else ""
    print(f"  {speaker}: {per_speaker[speaker]:.3f}   [{detail}]{spread}")

overall = sum(per_speaker.values()) / len(per_speaker)
gap = max(per_speaker.values()) - min(per_speaker.values())
print(f"\n전체 평균 {overall:.3f} · 화자 간 편차 {gap:.3f}")

## 10. 체크포인트 확인

In [ ]:
checkpoint = torch.load(DRIVE_CHECKPOINTS / "best.pt", map_location="cpu")

print(f"에폭 {checkpoint['epoch']}")
print(f"클래스 {checkpoint['num_classes']}개")
print(f"검증 정확도 {checkpoint['val_accuracy']:.3f}")
print(f"최근 평균 {checkpoint['smoothed_accuracy']:.3f}")
print(
    f"모델 hidden {checkpoint['hidden_dim']} · "
    f"layer {checkpoint['num_layer']} · dropout {checkpoint['dropout']}"
)

In [ ]:
from pathlib import Path
import shutil, numpy as np

local = Path(TRAIN_ROOT) / "processed"

drive_names = {p.name for p in DRIVE_PROCESSED.glob("*.npy")}
local_names = {p.name for p in local.glob("*.npy")}
missing = drive_names - local_names
broken  = {p.name for p in local.glob("*.npy") if p.stat().st_size == 0}
fix = missing | broken

print(f"누락 {len(missing)} · 0바이트 {len(broken)} · 복구 대상 {len(fix)}개")
for name in sorted(fix):
    src = DRIVE_PROCESSED / name
    shutil.copy2(src, local / name)
    print(f"  {name}  ({src.stat().st_size:,}바이트)")

# 검증
bad = []
for p in local.glob("*.npy"):
    try:
        if p.stat().st_size == 0:
            raise ValueError
        np.load(p, mmap_mode="r")
    except Exception:
        bad.append(p.name)

print(f"\n로컬 {len(list(local.glob('*.npy')))}개 / Drive {len(drive_names)}개 · 손상 {len(bad)}개")
assert len(bad) == 0 and len(local_names | fix) == len(drive_names), "아직 안 맞습니다"

In [ ]:
import torch, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from src.ml.models.lip_reading_model import LipReadingModel
from src.ml.training.dataset import LipReadingDataset
from src.ml.training.train import split_by_speaker

SPEAKERS = ["s01", "s03", "s04", "s05", "s06", "s07", "s08", "s09"]
TAU_MAIN = 1.0          # 사전 선택값. 이걸로 판정한다

def probs_fold(speaker):
    ds = LipReadingDataset(manifest_path, TRAIN_ROOT)
    _, vi, _ = split_by_speaker(ds, val_speakers=[speaker])
    loader = DataLoader(Subset(ds, vi), batch_size=16, num_workers=2)
    ck = torch.load(DRIVE_CHECKPOINTS / f"cv_{speaker}_seed42.pt", map_location="cuda")
    m = LipReadingModel(num_classes=ck["num_classes"], hidden_dim=ck["hidden_dim"],
                        num_layer=ck["num_layer"], dropout=ck["dropout"]).cuda()
    m.load_state_dict(ck["model_state"]); m.eval()
    P, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                o = m(x.cuda())
            P.append(F.softmax(o.float(), 1).cpu()); Y.append(y)
    return torch.cat(P), torch.cat(Y), sorted({r["label_text"] for r in ds.rows})

fold = {}
for sp in SPEAKERS:
    fold[sp] = probs_fold(sp)
    print(f"{sp} 완료")

texts = fold["s01"][2]
C = len(texts)
total = sum(len(fold[s][1]) for s in SPEAKERS)

# ── 1. 예측 분포가 얼마나 치우쳤나 ──
pred_n, true_n = torch.zeros(C), torch.zeros(C)
for P, Y, _ in fold.values():
    pred_n += torch.bincount(P.argmax(1), minlength=C).float()
    true_n += torch.bincount(Y, minlength=C).float()

print(f"\n{'문구':<16}{'정답':>6}{'예측':>6}{'배율':>7}")
print("-" * 37)
for i in torch.argsort(pred_n / true_n, descending=True):
    print(f"{texts[i]:<16}{int(true_n[i]):>6}{int(pred_n[i]):>6}{pred_n[i] / true_n[i]:>7.2f}")

# ── 2. 보정 ──
def freq(speakers):
    n = torch.zeros(C)
    for s in speakers:
        n += torch.bincount(fold[s][0].argmax(1), minlength=C).float()
    return (n / n.sum()).clamp(min=1e-6)

base = sum((fold[s][0].argmax(1) == fold[s][1]).sum().item() for s in SPEAKERS) / total

print(f"\n{'τ':>5}{'상한(자기 화자)':>16}{'정직(타화자 추정)':>18}")
print("-" * 40)
for tau in (0.0, 0.25, 0.5, 0.75, 1.0):
    hi = ho = 0
    for sp in SPEAKERS:
        P, Y, _ = fold[sp]
        others = [s for s in SPEAKERS if s != sp]
        hi += ((P / freq([sp]) ** tau).argmax(1) == Y).sum().item()
        ho += ((P / freq(others) ** tau).argmax(1) == Y).sum().item()
    mark = "  ← 판정" if tau == TAU_MAIN else ""
    print(f"{tau:>5.2f}{hi / total:>16.3f}{ho / total:>18.3f}{mark}")

print(f"\n보정 없음  {base:.3f}")

# ── 3. τ=1 정직 버전, 화자별 ──
print(f"\n{'화자':<6}{'보정전':>8}{'보정후':>8}{'차이':>8}")
print("-" * 32)
for sp in SPEAKERS:
    P, Y, _ = fold[sp]
    others = [s for s in SPEAKERS if s != sp]
    b = (P.argmax(1) == Y).float().mean().item()
    a = ((P / freq(others) ** TAU_MAIN).argmax(1) == Y).float().mean().item()
    print(f"{sp:<6}{b:>8.3f}{a:>8.3f}{a - b:>+8.3f}")

In [ ]:
import collections

for target_name in ["도와주세요", "자세바꿔주세요", "숨쉬기힘들어요"]:
    t = texts.index(target_name)
    print(f"\n=== {target_name} ===")
    print(f"{'화자':<6}{'정답':>6}{'맞춤':>6}{'재현율':>8}  주요 오답")
    for sp in SPEAKERS:
        P, Y, _ = fold[sp]
        mask = Y == t
        n = int(mask.sum())
        if n == 0:
            continue
        pred = P[mask].argmax(1)
        hit = int((pred == t).sum())
        wrong = collections.Counter(texts[int(i)] for i in pred if int(i) != t)
        top = " · ".join(f"{a}×{c}" for a, c in wrong.most_common(2))
        print(f"{sp:<6}{n:>6}{hit:>6}{hit / n:>8.2f}  {top}")


In [ ]:
import numpy as np, csv, collections
from pathlib import Path

with open(manifest_path, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

acc = collections.defaultdict(list)
for r in rows:
    a = np.load(Path(TRAIN_ROOT) / r["clip_path"])[:, :, :, 0].astype(np.float32)
    motion = np.abs(np.diff(a, axis=0)).mean()
    dup = np.mean([np.array_equal(a[i], a[i + 1]) for i in range(len(a) - 1)])
    acc[r["label_text"]].append((motion, dup))

print(f"{'문구':<16}{'움직임':>8}{'중복률':>8}{'클립':>6}")
print("-" * 40)
for m, ph, d, n in sorted((np.mean([x[0] for x in v]), ph,
                           np.mean([x[1] for x in v]), len(v))
                          for ph, v in acc.items()):
    print(f"{ph:<16}{m:>8.2f}{d:>8.2f}{n:>6}")

# 증강


In [ ]:
import importlib, sys
from src.ml.preprocess.augmentation import pipeline

fresh = importlib.reload(pipeline)                    # 소스에서 원본을 새로 읽음
patched = sys.modules["src.ml.training.train"].VideoAugmentation
patched.__call__ = fresh.VideoAugmentation.__call__   # 학습 코드가 쥔 클래스에 되돌림
print("복구:", patched.__call__.__qualname__)          # VideoAugmentation.__call__ 이면 정상

In [ ]:
# ═══ 시간축 증강 실험 · 이 셀 하나만 실행 (재실행 안전) ═══
from pathlib import Path
import numpy as np
from src.ml.preprocess.augmentation.pipeline import VideoAugmentation
from src.ml.training.train import train

DRIVE_ROOT        = globals().get("DRIVE_ROOT", Path("/content/drive/MyDrive/hanium-lipreading"))
DRIVE_CHECKPOINTS = globals().get("DRIVE_CHECKPOINTS", DRIVE_ROOT / "checkpoints")
manifest_path     = globals().get("manifest_path", DRIVE_ROOT / "manifest.csv")
TRAIN_ROOT        = globals().get("TRAIN_ROOT",
                    Path("/content/data") if Path("/content/data/processed").exists() else DRIVE_ROOT)
assert manifest_path.exists(), f"매니페스트 없음: {manifest_path}"

TIME_CROP_PROB = 0.5
TIME_CROP_MIN  = 0.75
SEEDS = [42, 1, 7]

# 원본을 클래스 속성에 한 번만 보관 → 몇 번 실행해도 진짜 원본이 유지된다
if not getattr(VideoAugmentation, "_taug_patched", False):
    VideoAugmentation._taug_orig = VideoAugmentation.__call__
    VideoAugmentation._taug_patched = True
ORIG = VideoAugmentation._taug_orig
assert ORIG.__qualname__ == "VideoAugmentation.__call__", f"원본이 아님: {ORIG.__qualname__}"

def call_with_time_aug(self, clip, return_details=False):
    if return_details:
        return ORIG(self, clip, True)
    frames = ORIG(self, clip, False)
    if self.rng.random() < TIME_CROP_PROB:
        T = len(frames)
        keep = int(T * self.rng.uniform(TIME_CROP_MIN, 1.0))
        if 2 <= keep < T:
            start = int(self.rng.integers(0, T - keep + 1))
            idx = np.linspace(start, start + keep - 1, T).round().astype(int)
            frames = frames[idx]
    return frames

VideoAugmentation.__call__ = call_with_time_aug
print(f"시간축 증강 적용 · 확률 {TIME_CROP_PROB} · 크롭 하한 {TIME_CROP_MIN}")
print(f"데이터 {TRAIN_ROOT}\n")

results = {}
try:
    for seed in SEEDS:
        print(f"\n{'='*16} s06 · seed {seed} · 시간축 증강 {'='*16}")
        results[seed] = train(
            manifest_path=manifest_path, data_root=TRAIN_ROOT,
            epochs=80, batch_size=16, learning_rate=2e-4,
            seed=seed, val_speakers=["s06"],
            checkpoint_path=DRIVE_CHECKPOINTS / f"taug_s06_seed{seed}.pt",
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name=f"taug_s06_seed{seed}",
        )
finally:
    VideoAugmentation.__call__ = ORIG
    print("\n[증강 원상복구 완료]")

v = [results[s] for s in SEEDS if s in results]
print(f"\n{'='*56}")
if len(v) == len(SEEDS):
    print(f"시간축 증강   {[f'{x:.3f}' for x in v]}   평균 {sum(v)/3:.3f} · 폭 {max(v)-min(v):.3f}")
else:
    print(f"완료 {len(v)}/{len(SEEDS)}시드   {[f'{x:.3f}' for x in v]}")
print(f"기준선(30)   ['0.732', '0.783', '0.745']   평균 0.754 · 폭 0.051")
print(f"채택선       0.805")

#오디오추출


In [21]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, time
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
PROCESSED_F60 = DRIVE_ROOT / "processed_f60"

n_drive = len(list(PROCESSED_F60.glob("*.npy")))
print("Drive processed_f60:", n_drive, "개")
assert n_drive == 1234, "processed_f60이 1234개가 아니다 - 전처리 이력 확인 필요"

REPO_DIR = Path("/content/hanium-lipreading")
os.chdir("/content")
if (REPO_DIR / ".git").exists():
    !cd {REPO_DIR} && git fetch origin && git checkout develop && git pull
else:
    !git clone -b develop https://github.com/HumanRhoid/hanium-lipreading.git {REPO_DIR\}
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
!pip install --quiet wandb

from scripts.build_manifest import build
manifest_f60 = DRIVE_ROOT / "manifest_f60.csv"
build(processed_dir=PROCESSED_F60, manifest_path=manifest_f60)

TRAIN_ROOT_F60 = Path("/content/data_f60")
LOCAL_F60 = TRAIN_ROOT_F60 / "processed"
started = time.time()
have = len(list(LOCAL_F60.glob("*.npy"))) if LOCAL_F60.exists() else 0
if have != n_drive:
    shutil.rmtree(TRAIN_ROOT_F60, ignore_errors=True)
    shutil.copytree(PROCESSED_F60, LOCAL_F60)

n_local = len(list(LOCAL_F60.glob("*.npy")))
bad = [p.name for p in LOCAL_F60.glob("*.npy") if p.stat().st_size == 0]
print("로컬", n_local, "개 ·", round(time.time() - started), "초 · 0바이트", len(bad), "개")
assert n_local == n_drive and not bad, "로컬 복사 불완전"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive processed_f60: 1234 개
Already on 'develop'
Your branch is up to date with 'origin/develop'.
Already up to date.
매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest_f60.csv
  클립 1234개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요
로컬 1234 개 · 0 초 · 0바이트 0 개


In [22]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, time, subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
PROCESSED_F60 = DRIVE_ROOT / "processed_f60"

n_drive = len(list(PROCESSED_F60.glob("*.npy")))
print("Drive processed_f60:", n_drive, "개")
assert n_drive == 1234, "processed_f60이 1234개가 아니다"

os.chdir("/content")
for junk in Path("/content").glob("*REPO_DIR*"):
    shutil.rmtree(junk, ignore_errors=True)
    print("잘못 만들어진 폴더 삭제:", junk.name)

REPO = "/content/hanium-lipreading"
REPO_DIR = Path(REPO)
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", REPO, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "develop"], check=True)
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git", "clone", "-b", "develop",
                    "https://github.com/HumanRhoid/hanium-lipreading.git", REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "wandb"], check=True)

from scripts.build_manifest import build
manifest_f60 = DRIVE_ROOT / "manifest_f60.csv"
build(processed_dir=PROCESSED_F60, manifest_path=manifest_f60)

TRAIN_ROOT_F60 = Path("/content/data_f60")
LOCAL_F60 = TRAIN_ROOT_F60 / "processed"
started = time.time()
have = len(list(LOCAL_F60.glob("*.npy"))) if LOCAL_F60.exists() else 0
if have != n_drive:
    shutil.rmtree(TRAIN_ROOT_F60, ignore_errors=True)
    shutil.copytree(PROCESSED_F60, LOCAL_F60)

n_local = len(list(LOCAL_F60.glob("*.npy")))
bad = [p.name for p in LOCAL_F60.glob("*.npy") if p.stat().st_size == 0]
print("로컬", n_local, "개 ·", round(time.time() - started), "초 · 0바이트", len(bad), "개")
assert n_local == n_drive and not bad, "로컬 복사 불완전"
# end$0

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive processed_f60: 1234 개
매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest_f60.csv
  클립 1234개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요
로컬 1234 개 · 0 초 · 0바이트 0 개


In [ ]:
from src.ml.training.train import train
from src.ml.preprocess.augmentation import VideoAugmentation

print("call:", VideoAugmentation.__call__.__qualname__)
assert VideoAugmentation.__call__.__qualname__ == "VideoAugmentation.__call__", "패치가 걸려 있다"

acc = train(
    manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
    epochs=80, batch_size=16, learning_rate=2e-4, seed=42,
    val_speakers=["s06"],
    checkpoint_path=DRIVE_CHECKPOINTS / "repro_s06_seed42.pt",
    num_workers=8, amp=True, ema_decay=0.998,
    hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
    wandb_project="lipreading", run_name="repro_s06_seed42")

print("")
print("재현 결과 ", round(acc, 3))
print("  08-19b   0.809")
print("  08-20    0.739")
if abs(acc - 0.739) < abs(acc - 0.809):
    print("  판정: 08-19b가 오염됨")
else:
    print("  판정: 08-20 8-fold가 오염됨")
# end$0

In [ ]:
from src.ml.training.train import train

base80 = [0.713, 0.815, 0.694]
res = []
for sd in (42, 1, 7):
    print("")
    print("=== s01 · seed", sd, "· 120에폭 ===")
    a = train(
        manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
        epochs=120, batch_size=16, learning_rate=2e-4, seed=sd,
        val_speakers=["s01"],
        checkpoint_path=DRIVE_CHECKPOINTS / ("ep120_s01_seed" + str(sd) + ".pt"),
        num_workers=8, amp=True, ema_decay=0.998,
        hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
        wandb_project="lipreading", run_name="ep120_s01_seed" + str(sd))
    res.append(round(a, 3))
    print("누적:", res)

m80 = sum(base80) / 3
m120 = sum(res) / 3
print("")
print("80에폭  ", base80, "평균", round(m80, 3), "폭", round(max(base80) - min(base80), 3))
print("120에폭 ", res, "평균", round(m120, 3), "폭", round(max(res) - min(res), 3))
print("차이", round(m120 - m80, 3))
# end

In [15]:
import json, time
from src.ml.training.train import train

SPEAKERS = ["s01", "s03", "s04", "s05", "s06", "s07", "s08", "s09"]
SEEDS = [42, 1, 7]

LOG = DRIVE_ROOT / "cv60e120_results.json"
done = json.loads(LOG.read_text()) if LOG.exists() else []
seen = [tuple(x[:2]) for x in done]
print("이미 끝난 런", len(done), "/ 24")

t0 = time.time()
for sp in SPEAKERS:
    for sd in SEEDS:
        if (sp, sd) in seen:
            continue
        print("")
        print("========", sp, "· seed", sd, "· 60프레임 · 120에폭 ========")
        acc = train(
            manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
            epochs=120, batch_size=16, learning_rate=2e-4, seed=sd,
            val_speakers=[sp],
            checkpoint_path=DRIVE_CHECKPOINTS / ("cv60e120_" + sp + "_seed" + str(sd) + ".pt"),
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name="cv60e120_" + sp + "_seed" + str(sd))
        done.append([sp, sd, round(acc, 4)])
        LOG.write_text(json.dumps(done))
        print("저장 ·", len(done), "/ 24 · 경과", round((time.time() - t0) / 60), "분")
# end

이미 끝난 런 17 / 24

======== s07 · seed 7 · 60프레임 · 120에폭 ========


lr,███████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▁▁▁▁▁
train/acc,▁▄▆▇████████████████████████████████████
train/loss,█▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▂▁▁▁▁▁▁▃▃▅▅▅▅▆▆▆▆▇▇▇▇▇▇█▇█▇▇▆▇▇▇▇▇▇▇▇▇▇
val/acc_smoothed,▁▁▁▂▁▁▁▁▂▄▆▆▆▇▇▇█▇██▇████▇▆▆▆▆▇▇▇▇▇▇▇▇▇▇
val/loss,▃▃▄▅▆█▇▆▄▄▂▂▂▂▂▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁
lr,2e-05
train/acc,1
train/loss,0.56073
val/acc,0.2807
val/acc_smoothed,0.2807


장치: cuda | 클래스: 15개
학습 1063개 · 검증 171개 클립 | 검증 화자 ['s07']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/120] train loss 2.7438 acc 0.076 | val loss 2.7086 acc 0.064 avg 0.064 | lr 2.00e-04
[  2/120] train loss 2.4816 acc 0.187 | val loss 2.7139 acc 0.064 avg 0.064 | lr 2.00e-04
[  3/120] train loss 2.0944 acc 0.374 | val loss 2.7241 acc 0.064 avg 0.064 | lr 2.00e-04
[  4/120] train loss 1.7373 acc 0.559 | val loss 2.7561 acc 0.058 avg 0.062 | lr 1.99e-04
[  5/120] train loss 1.3860 acc 0.713 | val loss 2.8194 acc 0.058 avg 0.060 | lr 1.99e-04
[  6/120] train loss 1.1621 acc 0.816 | val loss 2.9271 acc 0.058 avg 0.058 | lr 1.99e-04
[  7/120] train loss 0.9924 acc 0.875 | val loss 3.0687 acc 0.058 avg 0.058 | lr 1.98e-04
[  8/120] train loss 0.8582 acc 0.914 | val loss 3.2128 acc 0.058 avg 0.058 | lr 1.98e-04
[  9/120] train loss 0.8253 acc 0.920 |

lr,█████████▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁
train/acc,▁▃▄▆▇███████████████████████████████████
train/loss,█▅▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▂▁▁▂▄▅▅▆▆▆▇▇██▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▆▇▇▇
val/acc_smoothed,▁▁▁▁▁▁▃▅▅▆▆▇▇▇▇█████▆▆▆▆▆▆▇▇▆▇▆▆▆▆▇▆▆▇▇▇
val/loss,▂▂▂▃▄▇██▇▂▁▂▂▂▂▃▃▂▂▂▂▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▁
best_val_acc,0.28655
best_val_acc_smoothed,0.29045
lr,0
train/acc,1
train/loss,0.56074


저장 · 18 / 24 · 경과 19 분

======== s08 · seed 42 · 60프레임 · 120에폭 ========


...

lr,██████▇▇▇▇▆▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁
train/acc,▁▅▇▇▇███████████████████████████████████
train/loss,█▇▆▅▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▃▆▇▆▇████▇███▇▇▇▇▇██▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
val/acc_smoothed,▁▁▁▁▁▇▇▇▇▇██████▇▇▇█▇████▇▇▇███▇▇█▇▇▇▇▇▇
val/loss,▄▄▄▄█▄▃▂▁▁▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃
best_val_acc,0.43333
best_val_acc_smoothed,0.43556
lr,0
train/acc,1
train/loss,0.56053


저장 · 23 / 24 · 경과 116 분

======== s09 · seed 7 · 60프레임 · 120에폭 ========


장치: cuda | 클래스: 15개
학습 1084개 · 검증 150개 클립 | 검증 화자 ['s09']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/120] train loss 2.7387 acc 0.074 | val loss 2.7071 acc 0.067 avg 0.067 | lr 2.00e-04
[  2/120] train loss 2.5012 acc 0.178 | val loss 2.7114 acc 0.067 avg 0.067 | lr 2.00e-04
[  3/120] train loss 2.2260 acc 0.282 | val loss 2.7195 acc 0.067 avg 0.067 | lr 2.00e-04
[  4/120] train loss 1.8859 acc 0.469 | val loss 2.7497 acc 0.067 avg 0.067 | lr 1.99e-04
[  5/120] train loss 1.4973 acc 0.670 | val loss 2.8397 acc 0.067 avg 0.067 | lr 1.99e-04
[  6/120] train loss 1.2370 acc 0.780 | val loss 2.9897 acc 0.067 avg 0.067 | lr 1.99e-04
[  7/120] train loss 1.0347 acc 0.850 | val loss 3.1642 acc 0.067 avg 0.067 | lr 1.98e-04
[  8/120] train loss 0.9106 acc 0.895 | val loss 3.3389 acc 0.067 avg 0.067 | lr 1.98e-04
[  9/120] train loss 0.8287 acc 0.922 |

lr,█████████▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁
train/acc,▁▅▇▇████████████████████████████████████
train/loss,█▆▄▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▂▂▇████▇▇▇▇▇▇▇▇▇▆▇▇▇▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇
val/acc_smoothed,▁▁▁▁▁▅▇██████▇▇▇▇▇▇▇▇▇▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇
val/loss,▇█▆▄▃▁▁▁▁▂▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▁
best_val_acc,0.46667
best_val_acc_smoothed,0.46444
lr,0
train/acc,1
train/loss,0.56068


저장 · 24 / 24 · 경과 135 분


In [26]:
import sys, urllib.request, unicodedata, time
import numpy as np, cv2, torch
from pathlib import Path
from torch import nn

from src.ml.preprocess.lip_crop import create_landmarker, LIP_LANDMARKS, lip_openness
from src.ml.preprocess.normalize import trim_to_speech, resample_frames
from src.ml.models.lip_reading_model import LipReadingModel
from src.ml.models.backbone import LipReadingBackbone
from src.ml.models.temporal import TemporalBiGRU
from src.ml.models.classification_head import ClassificationHead
from src.ml.training.dataset import LipReadingDataset
from src.ml.training import train as T

AUX_DIM, PROJ_DIM, FRAMES = 44, 128, 60
OUT  = DRIVE_ROOT / "aux_f60.npz"
PART = DRIVE_ROOT / "aux_f60_partial.npz"
EXT  = [".mp4", ".avi", ".mov"]

if not OUT.exists():
    MODEL = Path("/content/hanium-lipreading/models/face_landmarker.task")
    if not MODEL.exists():
        MODEL.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task", str(MODEL))
        print("랜드마커 모델 내려받음")

    cand = []
    for p in sorted(DRIVE_ROOT.iterdir()):
        if not p.is_dir(): continue
        for d in [p] + [q for q in sorted(p.iterdir()) if q.is_dir()]:
            v = [q for q in d.iterdir() if q.suffix.lower() in EXT]
            if v: cand.append([len(v), d])
    cand.sort(reverse=True, key=lambda x: x[0])
    assert cand, "원본 영상을 Drive에서 못 찾았다"
    RAW_DIR = cand[0][1]
    print("RAW_DIR", RAW_DIR, cand[0][0], "개")

    vids = dict()
    for p in sorted(RAW_DIR.iterdir()):
        if p.suffix.lower() in EXT:
            vids[unicodedata.normalize("NFC", p.stem)] = p
    stems = [unicodedata.normalize("NFC", q.stem) for q in sorted(PROCESSED_F60.glob("*.npy"))]
    hit = [s for s in stems if s in vids]
    print("npy", len(stems), "개 · 원본 매칭", len(hit), "개")
    assert len(hit) > len(stems) * 0.95, "RAW_DIR 매칭 실패"

    names, feats = [], []
    if PART.exists():
        _p = np.load(PART, allow_pickle=False)
        names, feats = list(_p["names"]), list(_p["feats"])
        print("이어받기", len(names), "개")
    done = set(names)
    lmk = create_landmarker()
    t0 = time.time()
    try:
        for k, stem in enumerate(hit):
            if stem in done: continue
            cap = cv2.VideoCapture(str(vids[stem]))
            pts, ops = [], []
            while True:
                ok, fr = cap.read()
                if not ok: break
                h, w = fr.shape[:2]
                import mediapipe as mp
                res = lmk.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))
                if not res.face_landmarks: continue
                L = res.face_landmarks[0]
                pts.append([[L[i].x * w, L[i].y * h] for i in LIP_LANDMARKS])
                ops.append(lip_openness(L, w, h))
            cap.release()
            if len(pts) < 2: continue
            idx = resample_frames(trim_to_speech(list(range(len(pts))), ops), FRAMES)
            P = np.array(pts, dtype=np.float32)[idx]
            O = np.array(ops, dtype=np.float32)[idx]
            P = P - P.mean(axis=1, keepdims=True)
            scale = float(np.median(np.linalg.norm(P[:, 0] - P[:, 10], axis=1))) + 1e-6
            P = P / scale
            dur = np.full((FRAMES, 1), len(pts) / float(FRAMES), dtype=np.float32)
            feats.append(np.concatenate([P.reshape(FRAMES, 42), O.reshape(FRAMES, 1), dur], axis=1).astype(np.float32))
            names.append(stem)
            if len(names) % 100 == 0:
                np.savez(PART, names=np.array(names), feats=np.array(feats))
                print(len(names), "/", len(hit), "·", round(time.time() - t0), "초")
    finally:
        lmk.close()
    np.savez(OUT, names=np.array(names), feats=np.array(feats, dtype=np.float32))
    print("저장", OUT, len(names), "개 ·", round(time.time() - t0), "초")

_z = np.load(OUT, allow_pickle=False)
AUX_FEATS = _z["feats"].astype(np.float32)
AUX_INDEX = dict(zip(list(_z["names"]), range(len(_z["names"]))))
print("보조 특징", AUX_FEATS.shape)

if not hasattr(LipReadingDataset, "_orig_getitem"):
    LipReadingDataset._orig_getitem = LipReadingDataset.__getitem__
if not hasattr(LipReadingModel, "_orig_init"):
    LipReadingModel._orig_init = LipReadingModel.__init__
    LipReadingModel._orig_forward = LipReadingModel.forward
if not hasattr(T, "_orig_run_epoch"):
    T._orig_run_epoch = T.run_epoch

MISS = []

def getitem_aux(self, index):
    frames, label = LipReadingDataset._orig_getitem(self, index)
    name = unicodedata.normalize("NFC", Path(self.rows[index]["clip_path"]).stem)
    j = AUX_INDEX.get(name, -1)
    if j < 0:
        MISS.append(name)
        a = np.zeros((frames.shape[1], AUX_DIM), dtype=np.float32)
    else:
        a = AUX_FEATS[j]
    return frames, torch.from_numpy(a), label

def init_aux(self, num_classes, hidden_dim=256, num_layer=2, dropout=0.2,
             pretrained=False, freeze_backbone=False):
    nn.Module.__init__(self)
    self.backbone = LipReadingBackbone(pretrained=pretrained)
    if freeze_backbone:
        self.backbone.freeze_resnet()
    self.aux_proj = nn.Sequential(nn.Linear(AUX_DIM, PROJ_DIM), nn.ReLU())
    self.temporal = TemporalBiGRU(
        input_dim=self.backbone.feature_dim + PROJ_DIM,
        hidden_dim=hidden_dim, num_layer=num_layer, dropout=dropout)
    self.head = ClassificationHead(
        input_dim=self.temporal.output_dim, num_classes=num_classes, dropout=dropout)

def forward_aux(self, frames, aux):
    f = self.backbone(frames)
    f = torch.cat([f, self.aux_proj(aux)], dim=2)
    return self.head(self.temporal(f))

def run_epoch_aux(model, loader, criterion, device, optimizer=None,
                  amp=False, grad_clip=None, averager=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    with torch.set_grad_enabled(is_training):
        for frames, aux, labels in loader:
            frames = frames.to(device, non_blocking=True)
            aux = aux.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp):
                logits = model(frames, aux)
                loss = criterion(logits, labels)
            if is_training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if grad_clip:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if averager is not None:
                    averager.update(model)
            total_loss += loss.float().item() * labels.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_count += labels.size(0)
    return total_loss / total_count, total_correct / total_count

LipReadingDataset.__getitem__ = getitem_aux
LipReadingModel.__init__ = init_aux
LipReadingModel.forward = forward_aux
T.run_epoch = run_epoch_aux
print("패치 적용 완료")
# end

랜드마커 모델 내려받음
RAW_DIR /content/drive/MyDrive/hanium-lipreading/raw 1234 개
npy 1234 개 · 원본 매칭 1234 개
100 / 1234 · 178 초
200 / 1234 · 359 초
300 / 1234 · 588 초
400 / 1234 · 696 초
500 / 1234 · 808 초
600 / 1234 · 936 초
700 / 1234 · 1076 초
800 / 1234 · 1321 초
900 / 1234 · 1749 초
1000 / 1234 · 1985 초
1100 / 1234 · 2134 초
1200 / 1234 · 2306 초
저장 /content/drive/MyDrive/hanium-lipreading/aux_f60.npz 1234 개 · 2365 초
보조 특징 (1234, 60, 44)
패치 적용 완료


In [27]:
res = []
for sp in ["s05", "s06"]:
    print("")
    print("======== 랜드마크 추가 ·", sp, "· seed 42 ========")
    acc = T.train(
        manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
        epochs=80, batch_size=16, learning_rate=2e-4, seed=42,
        val_speakers=[sp],
        checkpoint_path=DRIVE_CHECKPOINTS / ("lm_" + sp + ".pt"),
        num_workers=8, amp=True, ema_decay=0.998,
        hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
        wandb_project="lipreading", run_name="lm_" + sp)
    res.append([sp, round(acc, 4)])
    print("누적:", res)
print("")
print("좌표 못 찾은 클립:", len(set(MISS)))
# end



======== 랜드마크 추가 · s05 · seed 42 ========


lr,██████▇▇▇▇▇▆▆▆▆▅▅▄▄▄▃▃▂▂▁▁
train/acc,▁▂▃▅▆▇▇▇██████████████████
train/loss,█▇▆▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▂▂▂▂▂▂▂▂▂▂▁▁▂▂▂▃▄▄▅▅██████
val/acc_smoothed,▂▂▂▂▂▂▂▂▂▂▁▁▁▁▂▂▃▄▄▄▆▇████
val/loss,▁▁▁▁▂▃▄▅▅▆▆▇▇███▇▇▆▆▅▅▄▄▄▃
lr,0.00015
train/acc,0.99815
train/loss,0.57682
val/acc,0.18667
val/acc_smoothed,0.18444


장치: cuda | 클래스: 15개
학습 1084개 · 검증 150개 클립 | 검증 화자 ['s05']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.55M / 전체 14.55M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7407 acc 0.084 | val loss 2.7129 acc 0.067 avg 0.067 | lr 2.00e-04
[  2/80] train loss 2.4425 acc 0.181 | val loss 2.7174 acc 0.027 avg 0.047 | lr 2.00e-04
[  3/80] train loss 2.1478 acc 0.343 | val loss 2.7449 acc 0.007 avg 0.033 | lr 1.99e-04
[  4/80] train loss 1.7454 acc 0.548 | val loss 2.7993 acc 0.067 avg 0.033 | lr 1.99e-04
[  5/80] train loss 1.2856 acc 0.772 | val loss 2.9190 acc 0.067 avg 0.047 | lr 1.98e-04
[  6/80] train loss 1.0742 acc 0.841 | val loss 3.1107 acc 0.067 avg 0.067 | lr 1.97e-04
[  7/80] train loss 0.9099 acc 0.903 | val loss 3.3667 acc 0.067 avg 0.067 | lr 1.96e-04
[  8/80] train loss 0.8256 acc 0.932 | val loss 3.5762 acc 0.067 avg 0.067 | lr 1.95e-04
[  9/80] train loss 0.7490 acc 0.955 | val loss

lr,█████▇▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁
train/acc,▁▂▃▇▇███████████████████████████████████
train/loss,█▇▅▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▃▁▃▃▃▃▄▅▄▃▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆██████▆▆▆▆▆
val/acc_smoothed,▂▁▂▂▂▂▂▂▂▃▃▄▄▄▄▄▅▄▄▄▅▅▅▅▅▅▅▅▅▅█████▅▅▅▅▅
val/loss,▁▁▁▂▅██▆▅▅▅▅▅▅▅▄▄▄▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃
best_val_acc,0.22
best_val_acc_smoothed,0.22
lr,0
train/acc,1
train/loss,0.56145


누적: [['s05', 0.22]]

======== 랜드마크 추가 · s06 · seed 42 ========


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s06']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.55M / 전체 14.55M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7186 acc 0.081 | val loss 2.7097 acc 0.051 avg 0.051 | lr 2.00e-04
[  2/80] train loss 2.4627 acc 0.169 | val loss 2.7161 acc 0.064 avg 0.057 | lr 2.00e-04
[  3/80] train loss 2.1689 acc 0.323 | val loss 2.7369 acc 0.064 avg 0.059 | lr 1.99e-04
[  4/80] train loss 1.8165 acc 0.512 | val loss 2.7757 acc 0.083 avg 0.070 | lr 1.99e-04
[  5/80] train loss 1.4875 acc 0.669 | val loss 2.8461 acc 0.083 avg 0.076 | lr 1.98e-04
[  6/80] train loss 1.2335 acc 0.773 | val loss 2.9695 acc 0.083 avg 0.083 | lr 1.97e-04
[  7/80] train loss 0.9995 acc 0.870 | val loss 3.1063 acc 0.083 avg 0.083 | lr 1.96e-04
[  8/80] train loss 0.8790 acc 0.919 | val loss 3.2208 acc 0.083 avg 0.083 | lr 1.95e-04
[  9/80] train loss 0.8072 acc 0.934 | val loss

lr,█████████▇▇▇▇▆▆▆▅▅▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁
train/acc,▁▄▆▇▇███████████████████████████████████
train/loss,█▆▅▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▂▅▅▆▇▇▇▇▇▇▇▇▇▇▇████████████████▇▇▇▇▇
val/acc_smoothed,▁▁▁▁▁▁▂▂▃▆▇▇▇▇▇▇▇▇▇▇████████████████▇▇▇▇
val/loss,▆▆▆▇▇█▇▆▅▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.78344
best_val_acc_smoothed,0.78344
lr,0
train/acc,1
train/loss,0.56218


누적: [['s05', 0.22], ['s06', 0.7834]]

좌표 못 찾은 클립: 0


In [28]:
res2 = []
for sp in ["s05", "s06"]:
    for sd in [1, 7]:
        print("")
        print("======== 랜드마크 ·", sp, "· seed", sd, "========")
        acc = T.train(
            manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
            epochs=80, batch_size=16, learning_rate=2e-4, seed=sd,
            val_speakers=[sp],
            checkpoint_path=DRIVE_CHECKPOINTS / ("lm_" + sp + "_seed" + str(sd) + ".pt"),
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name="lm_" + sp + "_seed" + str(sd))
        res2.append([sp, sd, round(acc, 4)])
        print("누적:", res2)
# end


======== 랜드마크 · s05 · seed 1 ========


장치: cuda | 클래스: 15개
학습 1084개 · 검증 150개 클립 | 검증 화자 ['s05']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.55M / 전체 14.55M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7160 acc 0.094 | val loss 2.7086 acc 0.067 avg 0.067 | lr 2.00e-04
[  2/80] train loss 2.4327 acc 0.188 | val loss 2.7142 acc 0.067 avg 0.067 | lr 2.00e-04
[  3/80] train loss 2.0833 acc 0.380 | val loss 2.7294 acc 0.067 avg 0.067 | lr 1.99e-04
[  4/80] train loss 1.6750 acc 0.588 | val loss 2.7767 acc 0.113 avg 0.082 | lr 1.99e-04
[  5/80] train loss 1.2980 acc 0.768 | val loss 2.8880 acc 0.067 avg 0.082 | lr 1.98e-04
[  6/80] train loss 1.0439 acc 0.849 | val loss 3.0572 acc 0.067 avg 0.082 | lr 1.97e-04
[  7/80] train loss 0.9162 acc 0.898 | val loss 3.2646 acc 0.067 avg 0.067 | lr 1.96e-04
[  8/80] train loss 0.7871 acc 0.952 | val loss 3.4429 acc 0.067 avg 0.067 | lr 1.95e-04
[  9/80] train loss 0.7496 acc 0.958 | val loss

lr,██████▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁
train/acc,▁▄▆▇▇███████████████████████████████████
train/loss,█▇▆▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▃▃▃▃▁▅▆▇▇▇▇▇▇▅▇██▅▅▅▅▅███▇▇▇▇█▇▇▇▇▇▇▇▇▇▇
val/acc_smoothed,▂▂▂▂▁▂▅▅▅▅▇▇▇▆▅▆█▆▅▅▅▅▆████▇▇▇▇█▇▇▇▇▇▇▇▇
val/loss,▁▁▂▃▄███▇▆▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃
best_val_acc,0.20667
best_val_acc_smoothed,0.20889
lr,0
train/acc,1
train/loss,0.56182


누적: [['s05', 1, 0.2067]]

======== 랜드마크 · s05 · seed 7 ========


장치: cuda | 클래스: 15개
학습 1084개 · 검증 150개 클립 | 검증 화자 ['s05']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.55M / 전체 14.55M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7081 acc 0.100 | val loss 2.7096 acc 0.067 avg 0.067 | lr 2.00e-04
[  2/80] train loss 2.3696 acc 0.222 | val loss 2.7162 acc 0.067 avg 0.067 | lr 2.00e-04
[  3/80] train loss 2.0224 acc 0.418 | val loss 2.7386 acc 0.067 avg 0.067 | lr 1.99e-04
[  4/80] train loss 1.5894 acc 0.637 | val loss 2.7920 acc 0.067 avg 0.067 | lr 1.99e-04
[  5/80] train loss 1.2271 acc 0.792 | val loss 2.9159 acc 0.067 avg 0.067 | lr 1.98e-04
[  6/80] train loss 1.0286 acc 0.856 | val loss 3.1398 acc 0.067 avg 0.067 | lr 1.97e-04
[  7/80] train loss 0.8785 acc 0.923 | val loss 3.3900 acc 0.067 avg 0.067 | lr 1.96e-04
[  8/80] train loss 0.7933 acc 0.945 | val loss 3.6600 acc 0.067 avg 0.067 | lr 1.95e-04
[  9/80] train loss 0.7346 acc 0.964 | val loss

...

lr,███████▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▄▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁
train/acc,▁▂▅▆▇███████████████████████████████████
train/loss,█▇▆▅▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▂▅▆▇▇▇█▇▇██▇██▇▇████████████████████
val/acc_smoothed,▁▁▁▁▁▂▂▃▅▆▇▇▇▇█▇▇▇▇▇████████████████████
val/loss,▆▆▆▆▇██▇▆▃▂▂▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.77707
best_val_acc_smoothed,0.77707
lr,0
train/acc,1
train/loss,0.56245


누적: [['s05', 1, 0.2067], ['s05', 7, 0.1933], ['s06', 1, 0.7771]]

======== 랜드마크 · s06 · seed 7 ========


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s06']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.55M / 전체 14.55M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7146 acc 0.103 | val loss 2.7085 acc 0.025 avg 0.025 | lr 2.00e-04
[  2/80] train loss 2.4067 acc 0.214 | val loss 2.7188 acc 0.070 avg 0.048 | lr 2.00e-04
[  3/80] train loss 2.0878 acc 0.369 | val loss 2.7472 acc 0.070 avg 0.055 | lr 1.99e-04
[  4/80] train loss 1.7805 acc 0.512 | val loss 2.7970 acc 0.083 avg 0.074 | lr 1.99e-04
[  5/80] train loss 1.4504 acc 0.677 | val loss 2.8996 acc 0.083 avg 0.079 | lr 1.98e-04
[  6/80] train loss 1.2656 acc 0.754 | val loss 3.0519 acc 0.083 avg 0.083 | lr 1.97e-04
[  7/80] train loss 1.0404 acc 0.840 | val loss 3.2395 acc 0.083 avg 0.083 | lr 1.96e-04
[  8/80] train loss 0.8837 acc 0.914 | val loss 3.3805 acc 0.083 avg 0.083 | lr 1.95e-04
[  9/80] train loss 0.8185 acc 0.928 | val loss

lr,████▇▇▇▇▇▇▆▆▆▆▆▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
train/acc,▁▂▅▇▇███████████████████████████████████
train/loss,█▇▆▄▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▂▃▄▄▆▇▇▇█████████████████████▇▇▇▇▇▇
val/acc_smoothed,▁▁▁▁▂▂▂▂▂▃▄▆▇▇▇▇▇███████████████████████
val/loss,▆▆▇▇█▇▅▄▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂
best_val_acc,0.72611
best_val_acc_smoothed,0.73036
lr,0
train/acc,1
train/loss,0.56232


누적: [['s05', 1, 0.2067], ['s05', 7, 0.1933], ['s06', 1, 0.7771], ['s06', 7, 0.7261]]


In [27]:
from src.ml.training import train as T
# end
import csv
(DRIVE_ROOT / "no_s05_results.json").unlink(missing_ok=True)
# end
rows = list(csv.DictReader(open(manifest_f60, encoding="utf-8")))
keep = [r for r in rows if r["speaker_id"] != "s05"]
manifest_no5 = DRIVE_ROOT / "manifest_f60_no_s05_3.csv"
with open(manifest_no5, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["clip_path", "label_id", "label_text", "speaker_id", "take"])
    w.writeheader()
    for r in keep: w.writerow(r)
print("클립", len(rows), "→", len(keep), "· 문구", len(set(r["label_text"] for r in keep)), "· 화자", len(set(r["speaker_id"] for r in keep)))

LOG5 = DRIVE_ROOT / "no_s05_results.json"
import json
done = json.loads(LOG5.read_text()) if LOG5.exists() else []
seen = [tuple(x[:2]) for x in done]
for sp in ["s01", "s03", "s04", "s06", "s07", "s08", "s09"]:
    for sd in [1, 7]:
        if (sp, sd) in seen: continue
        print("")
        print("======== s05 제외 ·", sp, "· seed", sd, "========")
        acc = T.train(
            manifest_path=manifest_no5, data_root=TRAIN_ROOT_F60,
            epochs=80, batch_size=16, learning_rate=2e-4, seed=sd,
            val_speakers=[sp],
            checkpoint_path=DRIVE_CHECKPOINTS / ("no5_" + sp + "_seed" + str(sd) + ".pt"),
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name="no5_" + sp + "_seed" + str(sd))
        done.append([sp, sd, round(acc, 4)])
        LOG5.write_text(json.dumps(done))
        print("누적:", done)
# end

클립 1234 → 1084 · 문구 15 · 화자 7

======== s05 제외 · s01 · seed 1 ========


lr,██████████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁
train/acc,▁▁▂▃▅▇▇█████████████████████████████████
train/loss,█▇▇▆▅▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▂▁▂▂▂▂▂▂▂▂▂▂▄▅▅▆▆▆▇▇▇▇███████████▇▇▇▇▇▇▇
val/acc_smoothed,▁▁▁▁▁▁▁▁▂▁▁▁▂▂▄▅▆▆▆▆▇▇▇▇███████████▇▇▇▇▇
val/loss,▆▆▆▆▆▇████▆▅▄▃▃▂▂▂▁▂▂▂▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃
lr,7e-05
train/acc,0.99815
train/loss,1.67902
val/acc,0.42
val/acc_smoothed,0.42222


장치: cuda | 클래스: 15개
학습 927개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.6928 acc 0.081 | val loss 2.7041 acc 0.064 avg 0.064 | lr 2.00e-04
[  2/80] train loss 2.5494 acc 0.211 | val loss 2.6991 acc 0.070 avg 0.067 | lr 2.00e-04
[  3/80] train loss 2.4016 acc 0.353 | val loss 2.7052 acc 0.070 avg 0.068 | lr 1.99e-04
[  4/80] train loss 2.2141 acc 0.563 | val loss 2.7287 acc 0.070 avg 0.070 | lr 1.99e-04
[  5/80] train loss 2.0433 acc 0.768 | val loss 2.7750 acc 0.051 avg 0.064 | lr 1.98e-04
[  6/80] train loss 1.9166 acc 0.869 | val loss 2.8307 acc 0.038 avg 0.053 | lr 1.97e-04
[  7/80] train loss 1.8574 acc 0.922 | val loss 2.8856 acc 0.070 avg 0.053 | lr 1.96e-04
[  8/80] train loss 1.7917 acc 0.962 | val loss 2.9358 acc 0.076 avg 0.062 | lr 1.95e-04
[  9/80] train loss 1.7778 acc 0.960 | val loss 

lr,███████▇▇▇▇▇▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
train/acc,▁▂▅▆▇███████████████████████████████████
train/loss,█▇▅▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▁▁▁▁▁▂▂▃▄▅▅▆▇▇▇███████████████████
val/acc_smoothed,▁▁▁▁▁▁▁▁▁▁▃▄▅▆▆▇▇▇▇▇████████████████████
val/loss,▆▆▆██▇▆▆▆▅▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.67516
best_val_acc_smoothed,0.67728
lr,0
train/acc,1
train/loss,1.66855


누적: [['s01', 1, 0.6752]]

======== s05 제외 · s01 · seed 7 ========


...

장치: cuda | 클래스: 15개
학습 936개 · 검증 148개 클립 | 검증 화자 ['s03']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.6915 acc 0.089 | val loss 2.7040 acc 0.101 avg 0.101 | lr 2.00e-04
[  2/80] train loss 2.5485 acc 0.198 | val loss 2.6990 acc 0.068 avg 0.084 | lr 2.00e-04
[  3/80] train loss 2.4561 acc 0.307 | val loss 2.7023 acc 0.068 avg 0.079 | lr 1.99e-04
[  4/80] train loss 2.2783 acc 0.500 | val loss 2.7199 acc 0.068 avg 0.068 | lr 1.99e-04
[  5/80] train loss 2.1041 acc 0.694 | val loss 2.7600 acc 0.068 avg 0.068 | lr 1.98e-04
[  6/80] train loss 1.9705 acc 0.823 | val loss 2.8264 acc 0.081 avg 0.072 | lr 1.97e-04
[  7/80] train loss 1.8787 acc 0.911 | val loss 2.9141 acc 0.068 avg 0.072 | lr 1.96e-04
[  8/80] train loss 1.8205 acc 0.946 | val loss 2.9974 acc 0.074 avg 0.074 | lr 1.95e-04
[  9/80] train loss 1.7853 acc 0.963 | val loss 

lr,████████▇▇▇▇▆▆▆▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train/acc,▁▄▅▆▇███████████████████████████████████
train/loss,█▅▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▂▁▁▁▁▁▁▁▁▁▁▁▂▂▃▅▆▆▆▇▇█████▇▇▇▆▆▅▅▅▅▆▅▆▆▆
val/acc_smoothed,▂▁▁▁▁▁▁▁▁▁▁▁▂▂▆▇▇███████▇▇▆▆▆▆▆▆▆▆▆▆▆▆▆▆
val/loss,▄▅▆▇█████▆▂▂▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
best_val_acc,0.46622
best_val_acc_smoothed,0.46622
lr,0
train/acc,1
train/loss,1.66864


누적: [['s01', 1, 0.6752], ['s01', 7, 0.6688], ['s03', 1, 0.4662]]

======== s05 제외 · s03 · seed 7 ========


장치: cuda | 클래스: 15개
학습 936개 · 검증 148개 클립 | 검증 화자 ['s03']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7011 acc 0.082 | val loss 2.7077 acc 0.068 avg 0.068 | lr 2.00e-04
[  2/80] train loss 2.5649 acc 0.173 | val loss 2.7048 acc 0.068 avg 0.068 | lr 2.00e-04
[  3/80] train loss 2.3984 acc 0.356 | val loss 2.7102 acc 0.068 avg 0.068 | lr 1.99e-04
[  4/80] train loss 2.1966 acc 0.615 | val loss 2.7317 acc 0.068 avg 0.068 | lr 1.99e-04
[  5/80] train loss 2.0411 acc 0.767 | val loss 2.7767 acc 0.068 avg 0.068 | lr 1.98e-04
[  6/80] train loss 1.9165 acc 0.877 | val loss 2.8363 acc 0.068 avg 0.068 | lr 1.97e-04
[  7/80] train loss 1.8499 acc 0.908 | val loss 2.9022 acc 0.068 avg 0.068 | lr 1.96e-04
[  8/80] train loss 1.8110 acc 0.946 | val loss 2.9629 acc 0.068 avg 0.068 | lr 1.95e-04
[  9/80] train loss 1.7813 acc 0.965 | val loss 

KeyboardInterrupt: 

In [25]:
import csv, unicodedata, numpy as np, torch
from torch import nn
from pathlib import Path

Z = np.load(DRIVE_ROOT / "aux_f60.npz", allow_pickle=False)
FEAT = dict(zip([unicodedata.normalize("NFC", n) for n in Z["names"]], Z["feats"].astype(np.float32)))
rows = list(csv.DictReader(open(DRIVE_ROOT / "manifest_f60.csv", encoding="utf-8")))
X, Y, S = [], [], []
for r in rows:
    stem = unicodedata.normalize("NFC", Path(r["clip_path"]).stem)
    if stem not in FEAT: continue
    X.append(FEAT[stem]); Y.append(int(r["label_id"])); S.append(r["speaker_id"])
X = torch.tensor(np.stack(X)); Y = torch.tensor(Y); S = np.array(S)
NC = int(Y.max()) + 1
print("클립", len(X), "· 차원", tuple(X.shape[1:]), "· 클래스", NC)

class AuxOnly(nn.Module):
    def __init__(self, nclass, dim=128):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(44, dim), nn.ReLU())
        self.gru = nn.GRU(dim, dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.head = nn.Linear(dim * 2, nclass)
    def forward(self, x):
        h, _ = self.gru(self.proj(x))
        return self.head(h.mean(dim=1))

BASE = dict(zip(["s01","s03","s04","s05","s06","s07","s08","s09"],
                [0.713, 0.554, 0.760, 0.193, 0.739, 0.363, 0.715, 0.407]))
dev = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS, BS, SEED = 80, 16, 42
print("")
print("화자   랜드마크단독(저장)  (마지막)   영상기준선   우연")
res = []
for sp in sorted(set(S)):
    tr, va = np.where(S != sp)[0], np.where(S == sp)[0]
    mu = X[tr].reshape(-1, 44).mean(0); sg = X[tr].reshape(-1, 44).std(0) + 1e-6
    xt, yt = ((X[tr] - mu) / sg).to(dev), Y[tr].to(dev)
    xv, yv = ((X[va] - mu) / sg).to(dev), Y[va].to(dev)
    torch.manual_seed(SEED); np.random.seed(SEED)
    m = AuxOnly(NC).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=2e-4, weight_decay=0.01)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    g = torch.Generator().manual_seed(SEED)
    hist, best = [], 0.0
    for ep in range(EPOCHS):
        m.train()
        for i in torch.randperm(len(xt), generator=g).split(BS):
            opt.zero_grad(set_to_none=True)
            loss = crit(m(xt[i]), yt[i]); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        sch.step()
        m.eval()
        with torch.no_grad():
            acc = (m(xv).argmax(1) == yv).float().mean().item()
        hist.append(acc)
        best = max(best, float(np.mean(hist[-3:])))
    res.append([sp, round(best, 4), round(hist[-1], 4)])
    print(" ", sp, "   ", round(best, 3), "        ", round(hist[-1], 3), "     ",
          BASE.get(sp, 0), "     0.067")
print("")
print("평균  저장", round(np.mean([r[1] for r in res]), 3),
      "· 마지막", round(np.mean([r[2] for r in res]), 3),
      "· 영상기준선", round(np.mean(list(BASE.values())), 3))
# end

클립 1234 · 차원 (60, 44) · 클래스 15

화자   랜드마크단독(저장)  (마지막)   영상기준선   우연
  s01     0.41          0.344       0.713      0.067
  s03     0.419          0.392       0.554      0.067
  s04     0.504          0.48       0.76      0.067
  s05     0.322          0.273       0.193      0.067
  s06     0.507          0.49       0.739      0.067
  s07     0.216          0.187       0.363      0.067
  s08     0.38          0.311       0.715      0.067
  s09     0.413          0.387       0.407      0.067

평균  저장 0.397 · 마지막 0.358 · 영상기준선 0.555


In [26]:
import csv, json, unicodedata, subprocess, numpy as np, torch
from pathlib import Path
from torch import nn
from src.ml.training.dataset import LipReadingDataset
from src.ml.training import train as T

TEACH = DRIVE_ROOT / "teacher_f60.npz"
RAW_DIR = DRIVE_ROOT / "raw"
WHISPER = "openai/whisper-small"
TAU, ALPHA = 2.0, 0.5

rows = list(csv.DictReader(open(DRIVE_ROOT / "manifest_f60.csv", encoding="utf-8")))
PH = [t for _, t in sorted(set((int(r["label_id"]), r["label_text"]) for r in rows))]
print("문구", len(PH), "개")

if not TEACH.exists():
    subprocess.run(["pip", "install", "--quiet", "transformers", "accelerate"], check=False)
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
    from transformers.modeling_outputs import BaseModelOutput
    dev = "cuda"
    proc = WhisperProcessor.from_pretrained(WHISPER)
    proc.tokenizer.set_prefix_tokens(language="korean", task="transcribe")
    wm = WhisperForConditionalGeneration.from_pretrained(WHISPER).to(dev).eval()

    seqs = [proc.tokenizer(p).input_ids for p in PH]
    L = max(len(s) for s in seqs)
    pad = proc.tokenizer.pad_token_id if proc.tokenizer.pad_token_id is not None else proc.tokenizer.eos_token_id
    lab = torch.full((len(PH), L), pad, dtype=torch.long)
    msk = torch.zeros(len(PH), L - 1)
    for i, s in enumerate(seqs):
        lab[i, :len(s)] = torch.tensor(s)
        msk[i, 3:len(s) - 1] = 1.0
    lab, msk = lab.to(dev), msk.to(dev)
    din, tgt = lab[:, :-1], lab[:, 1:]

    names, probs, nofail = [], [], 0
    tmp = Path("/content/_a.wav")
    for k, r in enumerate(rows):
        stem = unicodedata.normalize("NFC", Path(r["clip_path"]).stem)
        src = None
        for e in [".mp4", ".avi", ".mov"]:
            c = RAW_DIR / (stem + e)
            if c.exists(): src = c; break
        if src is None: continue
        subprocess.run(["ffmpeg", "-y", "-loglevel", "quiet", "-i", str(src),
                        "-ac", "1", "-ar", "16000", str(tmp)], check=False)
        if not tmp.exists() or tmp.stat().st_size < 2000:
            nofail += 1; continue
        import soundfile as sf
        wav, sr = sf.read(str(tmp))
        f = proc(wav, sampling_rate=16000, return_tensors="pt").input_features.to(dev)
        with torch.no_grad():
            enc = wm.model.encoder(f).last_hidden_state.repeat(len(PH), 1, 1)
            out = wm(decoder_input_ids=din, encoder_outputs=BaseModelOutput(last_hidden_state=enc))
            lp = torch.log_softmax(out.logits.float(), dim=-1)
            tok = lp.gather(2, tgt.unsqueeze(2)).squeeze(2)
            sc = (tok * msk).sum(1) / msk.sum(1)
            p = torch.softmax(sc / TAU, dim=0).cpu().numpy()
        names.append(stem); probs.append(p)
        tmp.unlink(missing_ok=True)
        if len(names) % 200 == 0: print(len(names), "/", len(rows))
    np.savez(TEACH, names=np.array(names), probs=np.array(probs, dtype=np.float32))
    print("교사 저장", len(names), "개 · 오디오 실패", nofail)

Z = np.load(TEACH, allow_pickle=False)
TP = dict(zip([unicodedata.normalize("NFC", n) for n in Z["names"]], Z["probs"]))
lid = dict(zip([unicodedata.normalize("NFC", Path(r["clip_path"]).stem) for r in rows],
               [int(r["label_id"]) for r in rows]))
top1 = np.mean([1.0 * (int(np.argmax(TP[s])) == lid[s]) for s in TP])
ent = np.mean([-float((TP[s] * np.log(TP[s] + 1e-9)).sum()) for s in TP])
print("")
print("교사 정확도", round(float(top1), 3), "· 평균 엔트로피", round(float(ent), 3),
      "· 균일분포 엔트로피", round(float(np.log(len(PH))), 3))

if not hasattr(LipReadingDataset, "_orig_getitem"):
    LipReadingDataset._orig_getitem = LipReadingDataset.__getitem__
if not hasattr(T, "_orig_run_epoch"):
    T._orig_run_epoch = T.run_epoch
UNIF = np.full(len(PH), 1.0 / len(PH), dtype=np.float32)

def getitem_kd(self, index):
    frames, label = LipReadingDataset._orig_getitem(self, index)
    stem = unicodedata.normalize("NFC", Path(self.rows[index]["clip_path"]).stem)
    return frames, torch.from_numpy(TP.get(stem, UNIF).copy()), label

def run_epoch_kd(model, loader, criterion, device, optimizer=None,
                 amp=False, grad_clip=None, averager=None):
    training = optimizer is not None
    model.train(training)
    tl = tc = tn = 0
    with torch.set_grad_enabled(training):
        for frames, soft, labels in loader:
            frames = frames.to(device, non_blocking=True)
            soft = soft.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp):
                logits = model(frames)
                ce = criterion(logits, labels)
                kd = -(soft * torch.log_softmax(logits.float(), dim=1)).sum(1).mean()
                loss = (1.0 - ALPHA) * ce + ALPHA * kd
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if grad_clip: nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if averager is not None: averager.update(model)
            tl += loss.float().item() * labels.size(0)
            tc += (logits.argmax(1) == labels).sum().item()
            tn += labels.size(0)
    return tl / tn, tc / tn

LipReadingDataset.__getitem__ = getitem_kd
T.run_epoch = run_epoch_kd
print("증류 패치 적용 · alpha", ALPHA, "· tau", TAU)

LOGK = DRIVE_ROOT / "kd_results.json"
done = json.loads(LOGK.read_text()) if LOGK.exists() else []
seen = [tuple(x[:2]) for x in done]
for sp in ["s01", "s03", "s04", "s05", "s06", "s07", "s08", "s09"]:
    if (sp, 42) in seen: continue
    print("")
    print("======== 증류 ·", sp, "· seed 42 ========")
    acc = T.train(
        manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
        epochs=80, batch_size=16, learning_rate=2e-4, seed=42,
        val_speakers=[sp],
        checkpoint_path=DRIVE_CHECKPOINTS / ("kd_" + sp + ".pt"),
        num_workers=8, amp=True, ema_decay=0.998,
        hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
        wandb_project="lipreading", run_name="kd_" + sp)
    done.append([sp, 42, round(acc, 4)])
    LOGK.write_text(json.dumps(done))
    print("누적:", done)
# end

문구 15 개


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

장치: cuda | 클래스: 15개
학습 1083개 · 검증 151개 클립 | 검증 화자 ['s08']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.6708 acc 0.106 | val loss 2.7065 acc 0.073 avg 0.073 | lr 2.00e-04
[  2/80] train loss 2.5586 acc 0.189 | val loss 2.7007 acc 0.066 avg 0.070 | lr 2.00e-04
[  3/80] train loss 2.4156 acc 0.349 | val loss 2.7021 acc 0.053 avg 0.064 | lr 1.99e-04
[  4/80] train loss 2.2064 acc 0.593 | val loss 2.7125 acc 0.066 avg 0.062 | lr 1.99e-04
[  5/80] train loss 2.0691 acc 0.722 | val loss 2.7306 acc 0.066 avg 0.062 | lr 1.98e-04
[  6/80] train loss 1.9453 acc 0.846 | val loss 2.7594 acc 0.066 avg 0.066 | lr 1.97e-04
[  7/80] train loss 1.8850 acc 0.888 | val loss 2.8017 acc 0.066 avg 0.066 | lr 1.96e-04
[  8/80] train loss 1.8283 acc 0.926 | val loss 2.8368 acc 0.066 avg 0.066 | lr 1.95e-04
[  9/80] train loss 1.7992 acc 0.943 | val loss

lr,████████▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁
train/acc,▁▅▆▇▇███████████████████████████████████
train/loss,█▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▂▄▄▄▅▆▆▆▇██████▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████
val/acc_smoothed,▁▁▁▁▂▃▄▃▄▄▄▅▅▅▆▆▇▇▇█████▇▇▇▇▇▇▇▇▇▇▇▇███▇
val/loss,▆▆▆▇▇██▇▆▅▄▅▄▄▄▃▂▂▂▁▁▁▁▁▁▁▂▁▁▁▁▂▂▁▁▁▁▁▁▁
best_val_acc,0.42384
best_val_acc_smoothed,0.42384
lr,0
train/acc,1
train/loss,1.67609


누적: [['s01', 42, 0.586], ['s03', 42, 0.5405], ['s04', 42, 0.7933], ['s05', 42, 0.1667], ['s06', 42, 0.8535], ['s07', 42, 0.3918], ['s08', 42, 0.4238]]

======== 증류 · s09 · seed 42 ========


장치: cuda | 클래스: 15개
학습 1084개 · 검증 150개 클립 | 검증 화자 ['s09']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7065 acc 0.077 | val loss 2.7057 acc 0.067 avg 0.067 | lr 2.00e-04
[  2/80] train loss 2.6080 acc 0.131 | val loss 2.6990 acc 0.013 avg 0.040 | lr 2.00e-04
[  3/80] train loss 2.5332 acc 0.223 | val loss 2.6965 acc 0.067 avg 0.049 | lr 1.99e-04
[  4/80] train loss 2.3990 acc 0.360 | val loss 2.6974 acc 0.067 avg 0.049 | lr 1.99e-04
[  5/80] train loss 2.2183 acc 0.590 | val loss 2.7024 acc 0.067 avg 0.067 | lr 1.98e-04
[  6/80] train loss 2.0638 acc 0.743 | val loss 2.7142 acc 0.067 avg 0.067 | lr 1.97e-04
[  7/80] train loss 1.9441 acc 0.840 | val loss 2.7357 acc 0.067 avg 0.067 | lr 1.96e-04
[  8/80] train loss 1.8716 acc 0.895 | val loss 2.7613 acc 0.073 avg 0.069 | lr 1.95e-04
[  9/80] train loss 1.8214 acc 0.937 | val loss

KeyboardInterrupt: 